# Deformable DETR

[Zhu et al. (2021), Deformable DETR: Deformable Transformers for End-to-End Object Detection](https://arxiv.org/abs/2010.04159)

DETRは集合予測によりアンカーやNMSを排除できた一方、2つの実用上の課題を抱えていた。

1. **収束が非常に遅い**：COCOで500 epochもの学習が必要（Faster R-CNNは数十epoch程度）
2. **小さい物体の検出が弱い**：Encoder/DecoderのSelf/Cross-Attentionが特徴マップの**全画素**に対して計算されるため、高解像度の特徴マップを使うと計算量・メモリが画素数の2乗で爆発し、実質的に低解像度の特徴マップしか使えない

**Deformable DETR** は、Attentionの計算対象を「全画素」から「学習された少数のサンプリング点」に絞る **Deformable Attention Module** を導入することで、この2つの課題を同時に解決する。

## Deformable Attention Module

通常のTransformerのAttentionは、あるクエリ $q$ に対して特徴マップ上の**全ての位置**とのAttention重みを計算する（$O(HW)$）。Deformable Attentionは、各クエリごとに参照点（reference point）$p_q$ の周辺にある **$K$ 個のサンプリング点だけ**を見る（$K=4$ 程度、$K \ll HW$）。

$$
\mathrm{DeformAttn}(q, p_q, x) = \sum_{m=1}^{M} W_m \left[ \sum_{k=1}^{K} A_{mqk} \cdot W_m' x(p_q + \Delta p_{mqk}) \right]
$$

- $x$：入力特徴マップ、$q$：クエリ特徴
- $m$：Attention head のインデックス（$M$個）
- $\Delta p_{mqk}$：参照点 $p_q$ からのサンプリング位置オフセット。**クエリ特徴から線形層で直接予測される学習対象**（固定パターンではない）
- $A_{mqk}$：$k$番目のサンプリング点に対するAttention重み（softmaxで正規化）
- $x(p_q + \Delta p_{mqk})$：非整数座標になりうるためbilinear補間で特徴を取り出す

通常のAttentionのように全画素に対する類似度計算（$QK^\top$）を行わず、少数のサンプリング点とその重みを直接回帰する点が本質的な違い。計算量は $O(HW)$ ではなく $O(HWK)$ 程度（$K$は定数）に抑えられ、高解像度・多スケールの特徴マップを扱えるようになる。

## Multi-scale Deformable Attention

Deformable AttentionはCNNのFPNのように、複数解像度の特徴マップ $\{x^l\}_{l=1}^{L}$ に対して自然に拡張できる。各クエリは、正規化された参照点を各レベルの特徴マップ上の対応する位置に投影し、レベルごとに $K$ 個ずつサンプリングして集約する。

$$
\mathrm{MSDeformAttn}(q, \hat{p}_q, \{x^l\}_{l=1}^{L}) = \sum_{m=1}^{M} W_m \left[ \sum_{l=1}^{L} \sum_{k=1}^{K} A_{mlqk} \cdot W_m' x^l(\phi_l(\hat{p}_q) + \Delta p_{mlqk}) \right]
$$

これにより、DETRでは別途必要だったFPNのような多スケール統合の仕組みを、Attention機構自体に組み込むことができ、小さい物体の検出精度が大きく改善する。

## Two-stage方式とIterative Bounding Box Refinement

Deformable DETRでは、収束をさらに速めるための工夫も導入されている。

- **Two-stage方式**：DETRのobject queryはランダム初期化された学習パラメータだが、Deformable DETRのTwo-stage版では、まずEncoderの出力から領域候補（region proposal）を生成し、そのスコア上位のものをDecoderのobject query（の初期値）として使う。Faster R-CNNのRPNに近い発想を、Transformerの枠組みに取り込んでいる
- **Iterative Bounding Box Refinement**：各Decoder層でボックス位置を予測し、それを次の層の参照点として渡すことで、層を重ねるごとにボックス位置を徐々に精緻化する（Cascade R-CNNに近い発想）

## 効果

原論文では、DETRに比べて**10倍少ない学習エポック数**で同等以上の精度に到達し、特に小さい物体（AP$_S$）のスコアが大きく改善したと報告されている。Deformable Attentionはその後、Vision分野の様々なTransformerベースモデル（後述のRT-DETRなど）でも標準的な構成要素として使われている。

なお、Deformable Attentionは通常PyTorchのcustom CUDA opとして実装されており、本ノートでは概念の説明にとどめ、実行デモは扱わない（`torchvision`・素の`torch.hub`の範囲では提供されていない）。

## 参考文献

- Zhu, X. et al. (2021). [Deformable DETR: Deformable Transformers for End-to-End Object Detection](https://arxiv.org/abs/2010.04159)
- [fundamentalvision/Deformable-DETR — GitHub](https://github.com/fundamentalvision/Deformable-DETR)